## **Cropland Area Estimation -- Bungoma County, Kenya (2025 season)**

This notebook operationalizes the stratified random sampling methodology from the [Sample-Based Map Accuracy and Area Estimation workshop](https://github.com/jowa-ea/EstellaKenyaSamplingClass), applied here to a real classified cropland map for the **2025 season** rather than a synthetic demo pair -- from sample design through to the final design-based area and accuracy estimate.

Author: Josef Wagner ([University of Strasbourg](https://www.unistra.fr/fr), [NASA Harvest](https://www.nasaharvest.org/)) jwagner@unistra.fr

### **Workflow**

1. Reproject the map to an auto-derived equal-area projection.
2. Set the accuracy threshold (target CV) for the cropland area estimate.
3. Draw a small, proportionally-allocated pilot sample and annotate it in STAC Notator (**round-1 checkpoint**).
4. Use the annotated pilot to compute Neyman priors, then the total sample size (N) and per-stratum allocation (n_h) needed to hit the target CV.
5. Draw the full sample with the *same random seed* as the pilot, so the pilot's units -- and their annotations -- are reused inside the bigger sample rather than redone. Only the units NOT already covered by the pilot (e.g. 500 of a 600-unit Neyman sample, if the pilot covered 100) get exported for a second annotation round (**round-2 checkpoint**).
6. Concatenate the pilot and round-2 annotations into one fully annotated sample.
7. Compute the design-based cropland area estimate, with uncertainty.
8. Compute map accuracy (overall, user's, producer's), with uncertainty.

Steps 3 and 5 each pause for **manual annotation in STAC Notator, outside this notebook** -- there's no way around that, since this map has no pre-existing reference data. Re-run the relevant cell once the annotated file is saved.

### **Setup**
First, install packages and import helper functions

In [ ]:
## The tricky part: installing gdal

# Update packages
!apt-get update -qq

# Install GDAL system libraries
!apt-get install -y gdal-bin libgdal-dev

import os
os.environ['CPLUS_INCLUDE_PATH'] = '/usr/include/gdal'
os.environ['C_INCLUDE_PATH'] = '/usr/include/gdal'

%pip install --upgrade pip -q
%pip install numpy pandas shapely fiona geopandas scikit-learn pyproj -q
# Optional: install GDAL Python bindings (match the system version)
%pip install gdal==$(gdal-config --version) -q

from osgeo import gdal, ogr, osr

print("GDAL version:", gdal.__version__)

In [ ]:
def in_colab():
  try:
    import google.colab
    return True
  except ImportError:
    return False

if in_colab():
  # git-lfs is required to fetch the real input_data/*.tif raster (LFS-tracked);
  # without it, the clone would only pull a small LFS pointer stub file.
  !apt-get install -y git-lfs -qq
  !git lfs install
  import os
  repo_dir = 'E2E_StacNotator_AreaEstimation_Training'
  repo_url = 'https://github.com/jowa-ea/E2E_StacNotator_AreaEstimation_Training.git'

  if os.path.basename(os.getcwd()) == repo_dir and os.path.isdir('.git'):
    # Re-running this cell in an already-warm runtime: a prior run's %cd
    # persists across cell re-executions within the same kernel, so we're
    # already sitting inside the clone -- checking for a same-named
    # subfolder (as opposed to checking whether we're already IN it) would
    # find nothing and clone again into ourselves, nesting the repo inside
    # itself. Just pull in place instead.
    !git pull
  elif os.path.isdir(repo_dir):
    # Cloned earlier in this runtime, but this cell hasn't cd'ed into it yet
    # (e.g. re-run after a kernel restart that kept the disk around).
    %cd {repo_dir}
    !git pull
  else:
    !git clone {repo_url}
    %cd {repo_dir}
else:
  print("Running locally - skipping git clone")

In [ ]:
# Import modules and packages
import os
import pandas as pd
import geopandas as gpd
from osgeo import gdal
pd.options.display.float_format = '{:.3f}'.format
gdal.UseExceptions()

import utils_ea_reprojection as ea
import utils_stratified_random_sampling as srs

In [ ]:
## Paths
base_path = os.getcwd() if in_colab() else '.'
input_data_path = os.path.join(base_path, 'input_data')
outputs_path = os.path.join(base_path, 'outputs')
os.makedirs(outputs_path, exist_ok=True)

crop_map_2025 = os.path.join(input_data_path, 'BungomaCropland2025.tif')  # EPSG:4326

#### **Step 1.** Reproject to an equal-area projection

>Pixel counting (hectares per stratum) and equal-probability stratified sampling both assume every pixel covers the same amount of ground area. In geographic coordinates (**EPSG:4326**) that's false: a degree of longitude covers less real ground distance near the poles than at the equator, so a fixed-size pixel in degrees does **not** represent a fixed amount of ground area across (or even within) the raster. Any hectare calculation or "uniformly sample a pixel" procedure done directly on it would be distorted -- unequally sized pixels get an unequal chance of selection, and area sums would be biased.
>
>Reprojecting to an **equal-area projection** removes this distortion: every pixel covers the same ground area everywhere in the raster, so pixel counts translate directly and unbiasedly into hectares, and every pixel gets an equal chance of being sampled.
>
>Rather than hardcoding a CRS for Bungoma specifically, the cell below **auto-derives** a Lambert Azimuthal Equal-Area (LAEA) projection centered on the raster's own bounding-box centroid -- so the same code is reusable for a new season or a new study region without a manual CRS lookup.

In [ ]:
# Auto-derive an equal-area (LAEA) projection centered on the map's own extent, and reproject to it.
# Nearest-neighbour resampling is required here: the raster is categorical (0/1 class codes),
# and any other resampling method would blend/interpolate those codes into meaningless values.
proj_str = ea.derive_ea_proj_string(
    crop_map_2025, out_proj_path=os.path.join(outputs_path, 'bungoma2025_ea_proj.txt')
)
print('Auto-derived equal-area projection:\n', proj_str)

crop_map_2025_ea = os.path.join(outputs_path, 'BungomaCropland2025_ea.tif')
ea.raster_to_ea(crop_map_2025, crop_map_2025_ea, proj_str, resampling_method='nearest')

#### **Step 2.** Strata, target stratum, and the accuracy threshold

>In sampling terminology, a subdivision of the population is a **stratum**. The map is a population of pixels, divided here into strata **non-cropland** (0) and **cropland** (1).
>
>The cell below is where **the user sets the accuracy threshold** for the cropland area estimate: `cv_target`, the target coefficient of variation (relative precision) of the area estimate, at a chosen `confidence` level. This single number is what drives how many sample units get requested in Step 4 -- a tighter (smaller) `cv_target` demands a larger sample.
>
>`seed` is also set here, once, and reused for **every** random draw in this notebook (pilot sample, full sample, shuffling) -- that single shared seed is what makes the pilot's sample units nest inside the full sample in Step 5.
>
>`id_col`/`true_col` name the columns this notebook expects back from an annotation tool; different tools may export the point-id or true-class columns under different names, so these are left as user-editable settings rather than hardcoded. `pilot_annotated_path`/`round2_annotated_path` are where the notebook looks for each round's annotated file -- **the user sets these to wherever the annotated files actually end up** (e.g. if downloaded from STAC Notator to a different folder than `outputs/`).
>
>`STRATUM_TRUE_LABELS` describes how the annotation tool codes classes in its own `true_col` output, which doesn't have to match the map's own `STRATUM_LABELS` coding -- numerically or otherwise (e.g. the tool might use its own numeric label ids, or -- as set here for STAC Notator's `stacnotator_label_name` export -- the label's own text spelling) -- it only has to describe the same classes, spelled identically. The notebook uses it to relabel every annotated `true_col` value onto the map's own 0/1 coding as soon as it's loaded, so a given number always means the same class in both `pred` and `true` before the two are ever compared.

In [ ]:
# Strata of interest
strata = [0, 1]
STRATUM_LABELS = {0: 'Non-cropland', 1: 'Cropland'}
target_stratum = 1  # cropland: the stratum the accuracy threshold below applies to

# How the annotation tool codes the SAME classes in its own true-label output --
# may use different keys than STRATUM_LABELS (numeric or text; STAC Notator's
# stacnotator_label_name column is text, matching the label spelling directly),
# but must describe the exact same set of classes, spelled identically. Used to
# relabel the annotated true label onto the map's own coding so pred/true always
# mean the same thing at the same numeric value.
STRATUM_TRUE_LABELS = {'Non-cropland': 'Non-cropland', 'Cropland': 'Cropland'}

# ---- User-defined parameters ----
cv_target = 0.05   # target relative precision (CV) of the cropland area estimate, e.g. 0.05 = 5%
confidence = 0.95  # confidence level for the area estimate's CI
pilot_n = 100       # number of sample units in the small pilot sample (Step 3)
seed = 2025         # random seed -- reused for EVERY draw below (pilot, full sample, shuffle) so pilot units nest inside the full sample (Step 5)

# Column names expected in annotation-tool exports -- change these two if your tool
# names the point-id / true-class columns differently on export. true_col is set
# to STAC Notator's own 'stacnotator_label_name' export column, so its raw export
# can be used directly as the annotated file with no renaming.
id_col = 'id'
true_col = 'stacnotator_label_name'

# Where to find each round's annotated file once STAC Notator work is done --
# update these paths if they're saved somewhere other than outputs/.
pilot_annotated_path = os.path.join(outputs_path, 'pilot_sample_annotated.csv')
round2_annotated_path = os.path.join(outputs_path, 'bungoma2025_sample_units_round2_annotated.csv')

#### **Step 3.** Pixel counts, and a small proportionally-allocated pilot sample

>Pixel counting gives each stratum's mapped area (`Area_ha`) and area weight (`Wi`), now computed on the equal-area raster from Step 1 so the hectare conversion is unbiased across the whole map.

In [ ]:
pixel_counts_csv = os.path.join(outputs_path, 'pixelcounts_bungoma2025.csv')
pixel_counts = srs.compute_pixel_counts(crop_map_2025_ea, strata=strata, output_csv=pixel_counts_csv)
pixel_counts

>Before the full sample can be sized with Neyman allocation, we need a *prior* estimate of each stratum's variance in classification correctness -- and that isn't known before any reference data has been collected. So we draw a small, **proportionally-allocated** stratified random sample of `pilot_n` units directly from the map. Proportional allocation is a neutral default here: the pilot's job is to inform the variances used for allocation, not to be optimally allocated itself.

In [ ]:
pilot_allocation_csv = os.path.join(outputs_path, 'pilot_allocation.csv')
pilot_allocation = srs.allocate_proportional(
    pixel_counts, n_total=pilot_n, min_allocation=0, output_csv=pilot_allocation_csv
)
print('Pilot allocation:', pilot_allocation)

pilot_gdf = srs.draw_samples_nested(
    crop_map_2025_ea, pilot_allocation, seed=seed, id_prefix='BGM25', v=True
)
pilot_gdf.head()

In [ ]:
# Shuffle row order first, then assign the exported `id` from that shuffled
# order -- assigning id before shuffling would let its value still betray each
# unit's original per-stratum draw order to an interpreter, even after the
# rows themselves were reordered.
pilot_gdf = srs.shuffle_samples(pilot_gdf, seed=seed)
pilot_gdf = srs.assign_ids(pilot_gdf, id_col=id_col, v=True)

# Export the pilot sample (reprojected to EPSG:4326) for photo-interpretation in STAC Notator
pilot_csv, pilot_geojson, _ = srs.export_sample_units(
    pilot_gdf,
    os.path.join(outputs_path, 'pilot_sample_for_annotation.csv'),
    os.path.join(outputs_path, 'pilot_sample_for_annotation.geojson'),
    stratum_labels=STRATUM_LABELS, id_col=id_col, v=True,
)

> **Checkpoint -- round-1 (pilot) annotation required.**
> Download `outputs/pilot_sample_for_annotation.csv` (or the `.geojson`) and interpret every point in **STAC Notator** against the best available imagery for the 2025 season, recording each unit's *true* class. Export the result as a CSV with at least an `id` column (matching the `id_col` setting above) and a `true_col` column -- with `true_col` set to `'stacnotator_label_name'`, this is STAC Notator's own label-name export column, so its raw export can be used directly with no renaming. Values are coded per `STRATUM_TRUE_LABELS` set in Step 2 -- they don't need to match `stratum`'s own 0/1 coding, as long as `STRATUM_TRUE_LABELS` documents the annotation tool's scheme (numeric or text). Save it to `pilot_annotated_path` (set in Step 2), then run the next cell.

In [ ]:
pilot_pred, pilot_true, pilot_annotated = srs.load_pilot_annotations(
    pilot_gdf, pilot_annotated_path, id_col=id_col, true_col=true_col,
    stratum_labels=STRATUM_LABELS, true_labels=STRATUM_TRUE_LABELS,
)
pilot_annotated.head()

#### **Step 4.** Neyman priors, total sample size (N), and per-stratum allocation (n_h)

>`compute_pilot_variances` turns the annotated pilot into a prior per-stratum standard deviation `Sh` (Neyman prior). `compute_neyman_sample_size` then combines `Sh` with the `cv_target`/`confidence` set in Step 2 to get the total sample size `n_tot` needed to hit that accuracy threshold, via Neyman allocation (Cochran, 1977; see also Song et al., 2017):
>
>$$n_{tot} = \frac{z^2 \left(\sum_h W_h S_h\right)^2}{E^2} \qquad\qquad n_h = n_{tot}\frac{W_h S_h}{\sum_h W_h S_h}$$
>
>where `Wh` is the stratum weight, `Sh` is the within-stratum standard deviation, `z` is the standard normal deviate for the chosen confidence level, and `E = cv_target * p_target` is the desired confidence-interval half-width on the target class's proportion. Because `E` already scales with `z`, the actual CV of the resulting estimator, `SE(p_hat)/p_target`, works out to `cv_target / z`: tightening the confidence level for the same `cv_target` requires a larger sample. `allocate_neyman` then splits `n_tot` across strata (`n_h`) by their contribution to overall variance -- large, uncertain strata get proportionally more units.

In [ ]:
Sh_csv = os.path.join(outputs_path, 'pilot_stratum_variances.csv')
Sh = srs.compute_pilot_variances(strata, pilot_pred, pilot_true, output_csv=Sh_csv, v=True)
Sh

In [ ]:
n_tot = srs.compute_neyman_sample_size(
    pixel_counts, Sh, cv_target=cv_target, confidence=confidence, target_stratum=target_stratum, v=True
)

In [ ]:
neyman_allocation_csv = os.path.join(outputs_path, 'neyman_allocation.csv')
neyman_allocation = srs.allocate_neyman(
    pixel_counts, Sh, n_tot, output_allocation_csv=neyman_allocation_csv, v=True
)

>Neyman allocation is computed independently per stratum from `Sh`, so for an unusual pilot -- or a loose `cv_target` -- it's technically possible for a stratum's Neyman allocation to come out *smaller* than how many pilot units already exist in that stratum. Step 5 needs `n_h` (full) &ge; `n_h` (pilot) in every stratum for the pilot to fully nest inside the full sample, so we take the elementwise max of the two allocations before drawing.

In [ ]:
allocation = srs.reconcile_allocation_with_pilot(neyman_allocation, pilot_allocation, v=True)

#### **Step 5.** Draw the full sample -- nested with the pilot -- and export only the new units for annotation

>`draw_samples_nested` gives each stratum its own independent random stream keyed on `(seed, stratum)`, rather than one stream shared across strata. That's what makes two allocations sharing the same `seed` **nest**: as long as the full sample requests at least as many units per stratum as the pilot did (guaranteed by the reconciliation above), the first units accepted for the full sample are -- pixel for pixel -- identical to the pilot's. A single shared RNG stream would *not* have this property, because consuming a different number of draws in one stratum shifts the random state every later stratum sees.

In [ ]:
full_gdf = srs.draw_samples_nested(crop_map_2025_ea, allocation, seed=seed, id_prefix='BGM25', v=True)

# Sanity check: every pilot unit must be present in the full sample. Matched on
# `_sample_key` (the stable per-stratum draw-order key), not `id_col` -- `id_col`
# hasn't been (re)assigned for full_gdf yet, and is reassigned independently per
# round anyway (see Step 5 below).
assert set(pilot_gdf['_sample_key']) <= set(full_gdf['_sample_key']), 'pilot sample did not nest inside the full sample'
print(f"{len(pilot_gdf)} pilot units confirmed nested inside the {len(full_gdf)}-unit full sample.")

>`draw_samples_nested` fills one stratum at a time, so its output is grouped by stratum. We shuffle the row order before export so an interpreter working through the CSV top to bottom isn't shown long runs of the same class -- this doesn't change which units were selected, only the order they're listed in. The exported `id` is then (re)assigned from that shuffled order, so it doesn't itself betray the original per-stratum grouping; internal matching between rounds (e.g. reusing the pilot's annotations here) instead relies on `_sample_key`, a separate identifier set once at draw time that survives shuffling and id reassignment.
>
>Two files come out of this step: a **master** file with all `n_tot` units (the pilot's annotations already filled in, for bookkeeping), and a **round-2** file with only the units the pilot didn't already cover -- e.g. if Neyman calls for 600 units and the pilot covered 100, only the other 500 get sent out for a second round of annotation. No one re-annotates the pilot's units.

In [ ]:
full_gdf = srs.shuffle_samples(full_gdf, seed=seed)
full_gdf = srs.assign_ids(full_gdf, id_col=id_col, v=True)

# Master export: all n_tot units, with the pilot's own annotations already filled in --
# kept as the bookkeeping file combined with the round-2 annotations in Step 6.
# `pilot_truth` is matched in on `_sample_key`, not `id_col`, since `id_col` was just
# reassigned above from this round's own shuffle and no longer matches the pilot's.
# `true_col` is passed explicitly so the carried-forward pilot label lands under
# the same column name used everywhere else in the pipeline.
master_csv, master_geojson, _ = srs.export_sample_units(
    full_gdf,
    os.path.join(outputs_path, 'bungoma2025_sample_units_master.csv'),
    os.path.join(outputs_path, 'bungoma2025_sample_units_master.geojson'),
    stratum_labels=STRATUM_LABELS, pilot_truth=pilot_annotated, id_col=id_col, true_col=true_col, v=True,
)

# Round-2 export: only the units NOT already annotated in the pilot.
round2_gdf = full_gdf[~full_gdf['_sample_key'].isin(pilot_annotated['_sample_key'])].copy()
round2_csv, round2_geojson, round2_df = srs.export_sample_units(
    round2_gdf,
    os.path.join(outputs_path, 'bungoma2025_sample_units_round2_for_annotation.csv'),
    os.path.join(outputs_path, 'bungoma2025_sample_units_round2_for_annotation.geojson'),
    stratum_labels=STRATUM_LABELS, id_col=id_col, v=True,
)
print(f"{len(pilot_annotated)} unit(s) already annotated from the pilot; {len(round2_gdf)} unit(s) exported for round-2 annotation.")
round2_df.head()

> **Checkpoint -- round-2 annotation required.**
> Download `outputs/bungoma2025_sample_units_round2_for_annotation.csv` (or the `.geojson`) -- only the units the pilot didn't already cover -- and interpret each in **STAC Notator**, adding a `true_col` column (`'stacnotator_label_name'`, coded per `STRATUM_TRUE_LABELS`, same as round 1). Save the result to `round2_annotated_path` (set in Step 2), then run the next cell.

#### **Step 6.** Concatenate the pilot and round-2 annotation rounds

>The master file already has the pilot's annotations filled in (Step 5 copied them in); this step only needs to fill in the remaining rows from the round-2 annotated file. `combine_annotation_rounds` fails loudly if any unit -- from either round -- ends up with no annotation, so a partially-annotated round-2 file can't silently produce an incomplete estimate.

In [ ]:
annotated_path = os.path.join(outputs_path, 'bungoma2025_sample_units_annotated.csv')
annotated_df = srs.combine_annotation_rounds(
    master_csv, round2_annotated_path, annotated_path, id_col=id_col, true_col=true_col,
    stratum_labels=STRATUM_LABELS, true_labels=STRATUM_TRUE_LABELS,
)
print(f'{len(annotated_df)} sample units, all annotated.')

pred, true, _ = srs.load_full_annotations(annotated_path, id_col=id_col, stratum_col='stratum', true_col=true_col)
annotated_df.head()

#### **Step 7.** Design-based area estimate

>`compute_stratified_random_sampling_metrics` cross-tabulates each unit's map stratum (`pred`) against its annotated stratum (`true`) and turns the sample counts into an unbiased area estimate per class, with standard error and 95% CI, in hectares and as a percentage of the estimated area (Olofsson et al., 2014, Eqs. 8-10).

In [ ]:
metrics_csv = os.path.join(outputs_path, 'area_estimates_bungoma2025.csv')
metrics = srs.compute_stratified_random_sampling_metrics(pixel_counts, pred, true, output_csv=metrics_csv, v=True)
metrics

#### **Step 8.** Map accuracy

>`compute_accuracy_metrics` computes overall accuracy plus each stratum's user's accuracy (Ui -- of the pixels the map calls this class, what fraction really are) and producer's accuracy (Pi -- of the pixels that really are this class, what fraction the map called it), each with SE and 95% CI (Olofsson et al., 2014, Eqs. 1-3, 5-7).

In [ ]:
accuracy_csv = os.path.join(outputs_path, 'accuracy_metrics_bungoma2025.csv')
accuracy_metrics, overall_accuracy = srs.compute_accuracy_metrics(pixel_counts, pred, true, output_csv=accuracy_csv, v=True)
accuracy_metrics

### **Result**

In [ ]:
cropland = metrics.loc[target_stratum]

print('=== 2025-season cropland area estimate, Bungoma County ===')
print(f"Cropland area: {cropland['Area_ha']:.0f} ha +/- {cropland['CI_Ha']:.0f} ha "
      f"({cropland['CI%'] * 100:.1f}% relative precision at {int(confidence * 100)}% confidence; "
      f"target was {cv_target * 100:.1f}%)")
print(f"Overall map accuracy: {overall_accuracy['O']:.3f} +/- {overall_accuracy['CI']:.3f}")

This is the final 2025-season cropland area estimate for Bungoma County and its design-based uncertainty -- the end of this training's pipeline, from a classified map with no existing reference data through to a defensible, sample-based area and accuracy estimate.